# Released ResStock timeseries: load tables without running models

**Purpose:** download the released ResStock timeseries directly from S3 (no models, no OpenStudio runs). Alternative to sections 2-4 of `resstock_models_main.ipynb`.

NREL already publishes the simulated 15-min load of every ResStock 2024 release 2 building (`timeseries_individual_buildings/by_state/upgrade=N/state=NY/<bldg_id>-N.parquet`, with `N` = `UPGRADE`; verified to exist for all 766 baseline IDs in `building_ids_metadata_2024.csv`, and for upgrade 4 for the applicable buildings). The cell below reads only the `timestamp` and `out.electricity.total.energy_consumption` (kWh per 15 min) columns of each file and writes the same three tables as section 4 of `resstock_models_main.ipynb` (`bldg_id`, `time`, `ts_elec`; annual: `bldg_id`, `annl_elec`).

Differences from running the models yourself (`resstock_models_main.ipynb`):
- **Weather:** released data uses the TMY3 weather of the release, not `Ithaca_2024.epw`.
- **Timestamps:** released data is for calendar year 2018 (interval-ending stamps, first `00:15`, last `2019-01-01 00:00`). The year is replaced by `TARGET_YEAR` (2024), so the last 15-min stamp is `2025-01-01 00:00`; 2018 has no Feb 29, so there is no leap day to remove. Weekdays match 2024 until Feb 28 and are shifted by one day afterwards.
- **Hourly table:** hours are labelled by `floor("h")` of the interval-ending stamps (same convention as the AMI hourly table in `ami_data_prep.ipynb`), so each building has 8761 hourly rows (the last one, `2025-01-01 00:00`, holds a single 15-min value).
- **Output files** get a suffix (`OUT_SUFFIX`, default `_released`) so they do not overwrite the files written by `resstock_models_main.ipynb`. Set it to `""` to replace them.
- The model download and simulation sections of the main notebook are not needed for this route.
- Set `UPGRADE` in the "Upgrade selection" cell (0 = baseline). It selects the S3 folder and file names, the ID list (`building_ids_metadata_2024[_upNN].csv`) and adds `_upNN` to the output files (e.g. `resstock_timeseries_all_15_released_up04.parquet`).
- Needs `data/base_files/building_ids_metadata_2024[_upNN].csv` (written by section 1 of notebook 1 for the same `UPGRADE`); the "Upgrade selection" cell stops with an error if it is missing.

## Setup
Assumes the project environment is already created and activated (`python -m pip install -r requirements.txt`). The two setup cells below are the same in every notebook of this project; run them first. Run notebooks from `grid_edge_fy25/notebooks/` (all paths are relative to it).

In [ ]:
# Preflight check: assumes the environment was created from requirements.txt
import importlib.util

required_modules = ['pandas', 'numpy', 'pyarrow']
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(
        "Missing Python packages: " + ", ".join(missing) +
        "\nCreate/activate the project environment and run: python -m pip install -r requirements.txt"
    )
print("Preflight OK: environment is ready.")


In [ ]:
# Common setup (identical in all notebooks of this project)
import ast
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

# Paths (relative to grid_edge_fy25/notebooks/)
current_path = Path(os.getcwd())
data_path = current_path / ".." / "data"
base_files = data_path / "base_files"        # building id lists, measures, weather, workflow
input_files = data_path / "input_files"      # NY metadata, AMI consumption
parquet_folder = data_path / "parquet"       # load tables written for the optimization
results_path = current_path / ".." / "results"
extract_dir = data_path / "resstock_models"  # bldgXXXXXXX folders (models + simulation output)

parquet_folder.mkdir(parents=True, exist_ok=True)


---
## Upgrade selection
Set `UPGRADE` to the same value used in `1_resstock_models_download_and_run.ipynb` (0 = baseline, 4 = ENERGY STAR heat pump with existing system as backup, ...). It drives the S3 folder (`upgrade=N/`), the file names (`<bldg_id>-N.parquet`), the building-ID list and the `_upNN` output tag.

In [ ]:
# ========================
# UPGRADE SELECTION (keep in sync with 1_resstock_models_download_and_run.ipynb)
# ========================
UPGRADE = 4          # 0 = baseline, 4 = ENERGY STAR heat pump with existing system as backup, ...

assert isinstance(UPGRADE, int) and UPGRADE >= 0, "UPGRADE must be a non-negative integer"
UPGRADE_TAG = "" if UPGRADE == 0 else f"_up{UPGRADE:02d}"   # baseline keeps the original names

BUILDING_IDS_CSV = base_files / f"building_ids_metadata_2024{UPGRADE_TAG}.csv"
if not BUILDING_IDS_CSV.exists():
    raise FileNotFoundError(
        f"{BUILDING_IDS_CSV.name} not found: run section 1 of 1_resstock_models_download_and_run.ipynb "
        f"with UPGRADE = {UPGRADE} first."
    )
print(f"UPGRADE = {UPGRADE} (tag '{UPGRADE_TAG}'), building IDs: {BUILDING_IDS_CSV.name}")


---
## Download and build the load tables

In [ ]:
# --- Released ResStock timeseries -> 15-min / hourly / annual load tables (no OpenStudio runs) ---
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow import fs

# ========================
# CONFIG
# ========================
TS_S3_PREFIX = ("oedi-data-lake/nrel-pds-building-stock/"
                "end-use-load-profiles-for-us-building-stock/2024/"
                "resstock_tmy3_release_2/timeseries_individual_buildings/"
                f"by_state/upgrade={UPGRADE}/state=NY/")   # UPGRADE from the 'Upgrade selection' cell
TS_REGION = "us-west-2"

# Buildings to fetch: default = county-filtered list from section 1 of 1_resstock_models_download_and_run.ipynb (same UPGRADE)
TS_ID_CSV = base_files / f"building_ids_metadata_2024{UPGRADE_TAG}.csv"
TS_BUILDING_IDS = pd.read_csv(TS_ID_CSV)["bldg_id"].astype(int).tolist()
# TS_BUILDING_IDS = [140567, 253852, 379341]   # <-- or set a custom list
TS_LIMIT = None          # e.g. 5 for a quick test run

SRC_COL = "out.electricity.total.energy_consumption"   # kWh per 15 min
SRC_YEAR = 2018          # calendar year of the released timeseries
TARGET_YEAR = 2024       # year written to the output tables
N_WORKERS = 8

TS_OUT_DIR = Path("../data/parquet")
OUT_SUFFIX = f"_released{UPGRADE_TAG}"   # e.g. _released (baseline) or _released_up04; "" would overwrite the outputs of notebook 1
WRITE_15MIN_CSV = False      # the 15-min CSV is ~1 GB for 766 buildings; parquet is always written

TS_OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_15_CSV  = TS_OUT_DIR / f"resstock_timeseries_all_15{OUT_SUFFIX}.csv"
OUT_15_PQ   = TS_OUT_DIR / f"resstock_timeseries_all_15{OUT_SUFFIX}.parquet"
OUT_HR_CSV  = TS_OUT_DIR / f"resstock_timeseries_all_hour{OUT_SUFFIX}.csv"
OUT_HR_PQ   = TS_OUT_DIR / f"resstock_timeseries_all_hour{OUT_SUFFIX}.parquet"
OUT_ANL_CSV = TS_OUT_DIR / f"resstock_timeseries_annual{OUT_SUFFIX}.csv"
OUT_ANL_PQ  = TS_OUT_DIR / f"resstock_timeseries_annual{OUT_SUFFIX}.parquet"

SCHEMA = pa.schema([("bldg_id", pa.int64()), ("time", pa.timestamp("ns")), ("ts_elec", pa.float64())])
_s3fs = fs.S3FileSystem(anonymous=True, region=TS_REGION)

# ========================
# HELPERS
# ========================
def shift_year(ts: pd.Series) -> pd.Series:
    """Replace the calendar year (not a day offset, which would create a Feb 29)."""
    return pd.to_datetime(pd.DataFrame({
        "year":   ts.dt.year - SRC_YEAR + TARGET_YEAR,
        "month":  ts.dt.month,
        "day":    ts.dt.day,
        "hour":   ts.dt.hour,
        "minute": ts.dt.minute,
    })).astype("datetime64[ns]")

def fetch_one(bid: int):
    """Returns (bldg_id, df15, df_hr, annual_kwh, status)."""
    try:
        tbl = pq.read_table(f"{TS_S3_PREFIX}{bid}-{UPGRADE}.parquet",
                            columns=["timestamp", SRC_COL], filesystem=_s3fs)
    except FileNotFoundError:
        return bid, None, None, 0.0, "missing_on_s3"
    except Exception as e:
        return bid, None, None, 0.0, f"error: {e}"

    raw = tbl.to_pandas()
    df15 = pd.DataFrame({
        "bldg_id": bid,
        "time":    shift_year(raw["timestamp"]),
        "ts_elec": raw[SRC_COL].astype("float64"),
    })
    df_hr = (df15.assign(time=df15["time"].dt.floor("h"))
                 .groupby("time", as_index=False)["ts_elec"].sum())
    df_hr.insert(0, "bldg_id", bid)
    return bid, df15, df_hr, float(df15["ts_elec"].sum()), "ok"

def append_csv(df: pd.DataFrame, path: Path):
    df.to_csv(path, index=False, mode="a", header=not path.exists())

# ========================
# RUN: parallel download, stream-write combined tables
# ========================
ids = TS_BUILDING_IDS[:TS_LIMIT] if TS_LIMIT else TS_BUILDING_IDS
print(f"Upgrade {UPGRADE}: fetching {len(ids)} buildings from s3://{TS_S3_PREFIX}")

for p in [OUT_15_CSV, OUT_HR_CSV, OUT_ANL_CSV]:   # CSVs are appended to, so start clean
    p.unlink(missing_ok=True)

annual_rows, failed = [], []
writer_15 = pq.ParquetWriter(str(OUT_15_PQ), SCHEMA, compression="snappy")
writer_hr = pq.ParquetWriter(str(OUT_HR_PQ), SCHEMA, compression="snappy")
try:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        for i, (bid, df15, df_hr, ann, status) in enumerate(pool.map(fetch_one, ids), 1):
            if status != "ok":
                failed.append((bid, status))
                print(f"  {bid} {status}")
                continue
            writer_15.write_table(pa.Table.from_pandas(df15, schema=SCHEMA, preserve_index=False))
            writer_hr.write_table(pa.Table.from_pandas(df_hr, schema=SCHEMA, preserve_index=False))
            if WRITE_15MIN_CSV:
                append_csv(df15, OUT_15_CSV)
            append_csv(df_hr, OUT_HR_CSV)
            annual_rows.append((bid, ann))
            if i % 50 == 0 or i == len(ids):
                print(f"  {i}/{len(ids)} done")
finally:
    writer_15.close()
    writer_hr.close()

anl_df = pd.DataFrame(annual_rows, columns=["bldg_id", "annl_elec"]).sort_values("bldg_id")
anl_df.to_csv(OUT_ANL_CSV, index=False)
anl_df.to_parquet(OUT_ANL_PQ, index=False)

print("\nSaved:")
print(f"  15-min  {OUT_15_PQ}" + (f"  (+ {OUT_15_CSV.name})" if WRITE_15MIN_CSV else ""))
print(f"  Hourly  {OUT_HR_PQ}  (+ {OUT_HR_CSV.name})")
print(f"  Annual  {OUT_ANL_PQ}  (+ {OUT_ANL_CSV.name})")
print(f"Buildings written: {len(annual_rows)}   Failed: {len(failed)} {failed[:10]}")
